In [1]:
import polars as pl
import numpy as np

from ohlc_dss_model.data.data_loader import load_parquet
from ohlc_dss_model.data.integrity import remove_incomplete_days
from ohlc_dss_model.data.tagging import intraday_session_tagging, session_tagging
from ohlc_dss_model.features.session_aggregation import aggregate_sessions
from ohlc_dss_model.utils.dt_utils import convert_to_timezone

Next is we need to model volatility in our features, problem is since we only have OHLC data and not tick data we will essentially need to estimate it, there are a couple of model such as Close to close, Parkinsons, Rogers Satchell, Yang Zhang.

Lets look at Close to close first as it is the most simplest model out of all, it measures volatility based on closing time of each day, which means what happens between those closing times doesn't really matter, this is a huge flaw as our model relies heavily on intraday movement so this model is out of the question

Another thing is we really are not supposed to calculate sigma_yz_t to include New york session at all since if we were to include New York session in the calculation at all it means we will be leaking the new york volatility into our features

In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2016-01-03       ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   ┆ 2016-01-04 ┆ Asia            │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2016-01-03       ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    ┆ 2016-01-04 ┆ Asia            │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

We will then aggregate the session OHLC such that one row represents one trading day

In [3]:
features = (
    df.group_by(["Session", "Intraday_Session"])
    .agg([
        pl.col("Open").first().alias("O"),
        pl.col("High").max().alias("H"),
        pl.col("Low").min().alias("L"),
        pl.col("Close").last().alias("C"),
    ])
    
    .pivot(
        index="Session",
        on="Intraday_Session",
        values=["O", "H", "L", "C"],
    )
    .sort("Session")
)
print(features.head(5))
print(features.shape)
print(features.columns)

shape: (5, 13)
┌────────────┬──────────┬─────────┬────────────┬───┬────────────┬──────────┬─────────┬────────────┐
│ Session    ┆ O_London ┆ O_Asia  ┆ O_New York ┆ … ┆ L_New York ┆ C_London ┆ C_Asia  ┆ C_New York │
│ ---        ┆ ---      ┆ ---     ┆ ---        ┆   ┆ ---        ┆ ---      ┆ ---     ┆ ---        │
│ date       ┆ f64      ┆ f64     ┆ f64        ┆   ┆ f64        ┆ f64      ┆ f64     ┆ f64        │
╞════════════╪══════════╪═════════╪════════════╪═══╪════════════╪══════════╪═════════╪════════════╡
│ 2016-01-04 ┆ 4529.75  ┆ 4592.5  ┆ 4498.75    ┆ … ┆ 4429.75    ┆ 4498.25  ┆ 4529.5  ┆ 4506.25    │
│ 2016-01-05 ┆ 4509.0   ┆ 4505.5  ┆ 4494.75    ┆ … ┆ 4457.25    ┆ 4494.75  ┆ 4509.25 ┆ 4481.5     │
│ 2016-01-06 ┆ 4445.0   ┆ 4482.0  ┆ 4396.25    ┆ … ┆ 4391.0     ┆ 4396.25  ┆ 4444.5  ┆ 4448.5     │
│ 2016-01-07 ┆ 4365.0   ┆ 4450.25 ┆ 4316.0     ┆ … ┆ 4283.0     ┆ 4316.75  ┆ 4365.0  ┆ 4293.25    │
│ 2016-01-08 ┆ 4339.75  ┆ 4298.0  ┆ 4337.0     ┆ … ┆ 4255.75    ┆ 4335.5   ┆ 4340.0  

One of the differences between these estimators is that some of them such as Rogers-Satchell and Parkinson ignores overnight gaps, now we need to determine if overnight gaps are material or not

In [4]:
gap = features.with_columns([
    ((pl.col("O_Asia") / pl.col("C_New York").shift(1)).log()).alias("Overnight_Gap"),
    ((pl.col("O_Asia") / pl.col("C_Asia")).log()).alias("Asia_Return"),
    ((pl.col("C_London") / pl.col("O_London")).log()).alias("London_Return"),
]).drop_nulls(["Overnight_Gap"])

for col, label in [
    ("Overnight_Gap", "Overnight Gap (C_NY -> O_Asia)"),
    ("Asia_Return", "Asia O to C return"),
    ("London_Return", "London O to C return"),
]:
    vals = gap[col].to_numpy()
    print(f"{label}")
    print(f"mean abs: {np.abs(vals).mean():.6f}")
    print(f"std: {vals.std():.6f}")
    print(f"p95 abs: {np.percentile(np.abs(vals), 95):.6f}")
    print(f"max abs: {np.abs(vals).max():.6f}")
    print()

Overnight Gap (C_NY -> O_Asia)
mean abs: 0.000920
std: 0.003011
p95 abs: 0.003638
max abs: 0.092055

Asia O to C return
mean abs: 0.003299
std: 0.005409
p95 abs: 0.010341
max abs: 0.049560

London O to C return
mean abs: 0.003093
std: 0.004711
p95 abs: 0.009561
max abs: 0.057329



Lets look at the mean, although overnight gaps have a lower mean compared to all of the session we also see that standard deviation are quite high, this means data are noisy and a coefficient of variation of ~ 3.3x meaning there will be time where gaps will be huge, take a look at max abs, overnight gaps have the largest, this can be caused by either news or covid for example, this is an example why we cannot really ignore this gaps, so the only estimator that fits our case are Yang-Zhang estimator.

Parkinsons assumes zero drift, but we need to see if asia or london does have drift at all if so then it violates this assumption.